# Gas Satations Points Visualization

This notebook visualizes the distribution of Gas stations across Spain.

In [ ]:
import polars as pl
import geopandas as gpd
import folium
from folium.plugins import MarkerCluster
import os

## 1. Load data

### 1.1 Load Charging Points data

In [ ]:
data_path = "../data/processed/gas_stations_proximity.parquet"
gas_stations = gpd.read_parquet(data_path)
gas_stations = gas_stations.to_crs(epsg=4326)
gas_stations

### 1.3 Load Road Segments

In [ ]:
import geopandas as gpd
# Paths
processed_path = '../data/processed/integrated_road_network.parquet'

gdf_processed = gpd.read_parquet(processed_path)

if gdf_processed.crs != 4326:
    gdf_processed = gdf_processed.to_crs(epsg=4326)

print(f"Loaded {len(gdf_processed)} Road Segments.")
gdf_processed.head()

## 2. Interactive Maps

We use **CartoDB Positron** tiles for a clean visual style. Below are two ways to visualize the EV charging infrastructure:

1. **Ultra Fast Charging Points and Road Segments**
2. **Ultra Fast Charging Points near Road Segments (<1km)**

### 2.1 Ultra Fast Charging Points and Road Segments

In [ ]:
# Individual Points Visualization (No Clustering)
m_roads_gas_stations = folium.Map(location=[40.4168, -3.7038], zoom_start=6, tiles="CartoDB Positron")

for row in gas_stations.itertuples():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=3,
        color="#3186cc",
        fill=True,
        fill_color="#3186cc",
        fill_opacity=0.7,
        tooltip=getattr(row, "site_name", "")
    ).add_to(m_roads_gas_stations)


# Add Processed Segments
folium.GeoJson(
    gdf_processed,
    name="Processed Network",
    style_function=lambda x: {'color': 'blue', 'weight': 2, 'opacity': 0.7},
).add_to(m_roads_gas_stations)

m_roads_gas_stations

### 2.2 Ultra Fast Charging Points 1km or closer to Road Segments